## Step 1: Parse the PDF into structured elements

In [1]:
from unstructured.partition.pdf import partition_pdf

elements = partition_pdf(
    filename="sample_report.pdf",
    strategy="hi_res",
    infer_table_structure=True,
    extract_images_in_pdf=True,
    extract_image_block_output_dir="./extracted_images"
)

for i, el in enumerate(elements):
    print(f"[{i}] {el.category} | page {el.metadata.page_number} | {str(el.text)[:60] if el.text else '[image]'}")

/Users/rifat/miniconda3/envs/llm/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
No languages specified, defaulting to English.
Loading weights: 100%|██████████| 367/367 [00:00<00:00, 10897.92it/s]


[0] Title | page 1 | Q3 2026 Financial Summary
[1] NarrativeText | page 1 | This report summarizes quarterly performance across all oper
[2] Title | page 2 | Regional Revenue Breakdown
[3] NarrativeText | page 2 | Revenue grew across all regions this quarter. The table belo
[4] Table | page 2 | Region Revenue ($M) Growth (%) APAC 4.2 8% EMEA 3.1 5% Ameri
[5] Image | page 2 | Figure 1: Q3 Revenue by Region ($M) Revenue ($M) Ww aN N APA
[6] FigureCaption | page 2 | Figure 1: Q3 Revenue by Region
[7] Title | page 3 | Headcount Overview
[8] NarrativeText | page 3 | Total headcount increased modestly this quarter, with most h
[9] Table | page 3 | Department Headcount Change Engineering 142 +12 Sales 58 +3 


In [2]:
print(elements[4].metadata.text_as_html)

<table><thead><tr><th>Region</th><th>Revenue ($M)</th><th>Growth (%)</th></tr></thead><tbody><tr><td>APAC</td><td>4.2</td><td>8%</td></tr><tr><td>EMEA</td><td>3.1</td><td>5%</td></tr><tr><td>Americas</td><td>5.6</td><td>6%</td></tr></tbody></table>


## Step 2: Enrich non-text elements (table summary + image caption)

In [3]:
def summarize_table(html):
    # Replace with real LLM call: llm.generate(f"Summarize this table: {html}")
    return "Table showing regional revenue and growth: APAC 4.2M (8%), EMEA 3.1M (5%), Americas 5.6M (6%)."

def caption_image(image_path):
    # Replace with real vision LLM call
    return "Bar chart titled Figure 1 showing Q3 revenue by region: APAC $4.2M, EMEA $3.1M, Americas $5.6M."

for el in elements:
    if el.category == "Table":
        el.metadata.generated_summary = summarize_table(el.metadata.text_as_html)
    if el.category == "Image":
        el.metadata.generated_caption = caption_image(el.metadata.image_path)

In [4]:
print(elements[4].metadata.generated_summary)
print(elements[5].metadata.generated_caption)

Table showing regional revenue and growth: APAC 4.2M (8%), EMEA 3.1M (5%), Americas 5.6M (6%).
Bar chart titled Figure 1 showing Q3 revenue by region: APAC $4.2M, EMEA $3.1M, Americas $5.6M.


## Step 3 : Convert enriched elements → LlamaIndex Documents, grouped by page/section

In [5]:
from llama_index.core import Document
from collections import defaultdict

page_text = defaultdict(list)

for el in elements:
    page = el.metadata.page_number
    if el.category == "Table":
        page_text[page].append(f"[Table]: {el.metadata.generated_summary}")
    elif el.category == "Image":
        page_text[page].append(f"[Figure]: {el.metadata.generated_caption}")
    elif el.text:
        page_text[page].append(el.text)

documents = [
    Document(text="\n\n".join(texts), metadata={"page": page})
    for page, texts in sorted(page_text.items())
]

print(documents[1].text)  # page 2 preview

Regional Revenue Breakdown

Revenue grew across all regions this quarter. The table below shows the breakdown by region, and Figure 1 below visualizes the same data as a bar chart. Americas posted the highest absolute revenue, while APAC saw the fastest growth rate.

[Table]: Table showing regional revenue and growth: APAC 4.2M (8%), EMEA 3.1M (5%), Americas 5.6M (6%).

[Figure]: Bar chart titled Figure 1 showing Q3 revenue by region: APAC $4.2M, EMEA $3.1M, Americas $5.6M.

Figure 1: Q3 Revenue by Region


## Step 4: Apply HierarchicalNodeParser

In [6]:
from llama_index.core.node_parser import HierarchicalNodeParser, get_leaf_nodes

node_parser = HierarchicalNodeParser.from_defaults(chunk_sizes=[1024, 256])
hierarchical_nodes = node_parser.get_nodes_from_documents(documents)
leaf_nodes = get_leaf_nodes(hierarchical_nodes)

print(f"Total nodes (parent+leaf): {len(hierarchical_nodes)}, leaf nodes: {len(leaf_nodes)}")

Total nodes (parent+leaf): 6, leaf nodes: 3


## Step 5: Store

In [ ]:
import chromadb
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.core import VectorStoreIndex, StorageContext
from llama_index.core.storage.docstore import SimpleDocumentStore
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core import Settings

chroma_client = chromadb.PersistentClient(path="./chroma_db")
chroma_collection = chroma_client.get_or_create_collection("pdf_report")

vector_store = ChromaVectorStore(chroma_collection=chroma_collection)

# Doc store still separate — holds full node content + parent/child relationships
docstore = SimpleDocumentStore()
docstore.add_documents(hierarchical_nodes)

storage_context = StorageContext.from_defaults(
    vector_store=vector_store,
    docstore=docstore
)

Settings.embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")
index = VectorStoreIndex(leaf_nodes, storage_context=storage_context)
## To get previously stored indexs
# index = VectorStoreIndex.from_vector_store(vector_store)

print("Stored", chroma_collection.count(), "vectors in ChromaDB")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 23147.89it/s]


Stored 3 vectors in ChromaDB


## Step 6: Retrieve with auto-merging

In [ ]:
from llama_index.core.retrievers import AutoMergingRetriever

query = "What was APAC's revenue growth?"

# Leaf-level only, no merging — for comparison
base_retriever = index.as_retriever(similarity_top_k=3)
leaf_results = base_retriever.retrieve(query)

print("=== Base retriever (leaf-only) ===")
for n in leaf_results:
    print(f"score={n.score:.4f} | {len(n.node.text)} chars")
    print(n.node.text[:200])
    print()

# Wraps the base retriever: looks up each matched leaf's PARENT relationship
# in the docstore, and swaps leaves for their parent chunk when enough
# siblings under that parent were also matched (simple_ratio_thresh, default 0.5)
merging_retriever = AutoMergingRetriever(base_retriever, storage_context, verbose=True)
merged_results = merging_retriever.retrieve(query)

print("=== Auto-merging retriever ===")
for n in merged_results:
    print(f"score={n.score:.4f} | {len(n.node.text)} chars")
    print(n.node.text[:300])
    print()

> Merging 1 nodes into parent node.
> Parent node id: 66f8325a-02a2-4517-a748-b3e7b48e4a83.
> Parent node text: Regional Revenue Breakdown

Revenue grew across all regions this quarter. The table below shows t...

> Merging 1 nodes into parent node.
> Parent node id: cf78d3e6-b3d8-415b-8115-f323fa10c59c.
> Parent node text: Q3 2026 Financial Summary

This report summarizes quarterly performance across all operating regi...

> Merging 1 nodes into parent node.
> Parent node id: 8713f4a6-109d-48e3-8284-746fb6d73b21.
> Parent node text: Headcount Overview

Total headcount increased modestly this quarter, with most hiring concentrate...

=== Auto-merging retriever ===
score=0.6326 | 510 chars
Regional Revenue Breakdown

Revenue grew across all regions this quarter. The table below shows the breakdown by region, and Figure 1 below visualizes the same data as a bar chart. Americas posted the highest absolute revenue, while APAC saw the fastest growth rate.

[Table]: Table showing regional 

